# Activation Steering

[Probing](linear-probes.ipynb) tells you a concept is *readable*. Steering asks the
question probing cannot: **is it used?** Add a direction to the residual stream at
inference time and see whether behaviour changes. If it does, you have a causal result
rather than a correlational one.

The technique is almost embarrassingly simple — compute the difference between the mean
activations of two contrasting prompt sets, then add that vector during generation — and
it works well enough to control refusal, sentiment, formality and factual assertions in
real models.

It is also the natural end of this track: probing finds candidate directions,
[SAEs](sparse-autoencoders.ipynb) find them without labels, and steering is how you test
whether any of them matter.

## 1. What & Why

The recipe, in full:

```
v = mean(activations on "positive" prompts) − mean(activations on "negative" prompts)
h' = h + α·v          # at some layer, during the forward pass
```

That is it. `v` is the **difference-in-means** direction, `α` is the strength.

**Why it works at all** rests on the *linear representation hypothesis*: that many
high-level concepts are encoded as directions in activation space, and that moving along
such a direction changes the model's behaviour in the corresponding way. Steering is
simultaneously an application of that hypothesis and a test of it — if adding a direction
reliably produces the expected behaviour change, the hypothesis has earned something.

**What it is good for:**

- **Causal verification.** The only cheap way to check that a direction found by a probe
  or an SAE actually drives behaviour.
- **Inference-time control** without fine-tuning: no gradients, no weight updates, one
  vector per behaviour, composable and reversible.
- **Safety-relevant analysis** — refusal, sycophancy and deception have all been
  localised to directions this way, and made stronger or weaker on demand.

**What it is not:** a robust control mechanism. Steering degrades output quality as
strength rises, generalises unpredictably off-distribution, and moves correlated concepts
along with the target one. It is a research instrument that sometimes doubles as a
product feature.

## 2. Mental Model

**A thumb on the scale, not a rewritten rulebook.**

Fine-tuning rewrites the model's dispositions. Steering leans on them at runtime: the
model still computes everything it would have computed, and you add a constant nudge in
one direction before the next layer reads it.

Three consequences follow, and each explains a characteristic behaviour:

- **The nudge is constant, but the activations are not.** The same `α·v` added to a
  confident activation is a small perturbation; added to an ambivalent one it is decisive.
  Steering therefore has its largest effect exactly where the model was undecided — which
  is usually what you want.
- **Push too hard and you leave the manifold.** Real activations occupy a small region of
  ℝᵈ. A large `α` puts you somewhere no training input ever produced, and the model's
  behaviour there is not merely wrong but unconstrained — degenerate repetition, broken
  syntax.
- **You are moving a direction, not a concept.** If two concepts share a direction —
  because they co-occurred in training — steering one moves the other, and there is
  nothing in the method that could distinguish them.

The framing that keeps expectations right: steering is a **causal probe you can also
deploy**, and its failure modes are the failure modes of assuming a concept is exactly one
direction.

## 3. Key Concepts

| Term | What it means |
|---|---|
| **Steering vector** | The direction added to activations. Usually difference-in-means over contrasting prompts. |
| **Difference-in-means (DIM)** | `mean(h⁺) − mean(h⁻)`. Simple, and empirically a better steering direction than probe weights. |
| **CAA** | Contrastive Activation Addition — the standard recipe, using paired prompts differing in one attribute. |
| **Steering coefficient `α`** | Strength. The whole quality/effect trade-off lives here. |
| **Injection layer** | Where to add. Middle layers usually work best; early is too raw, late is too committed. |
| **Off-manifold** | Activations no real input would produce. Where large `α` sends you, and where behaviour degenerates. |
| **Refusal direction** | The best-studied case: a single direction mediating refusal, whose ablation removes it. |
| **Ablation vs addition** | Projecting a direction *out* versus adding it *in*. Ablation is the cleaner causal test. |
| **Entanglement** | Correlated concepts sharing directions, so steering one moves the others. |
| **Probe weights vs DIM** | A probe finds *a* separating direction; DIM finds the direction the classes actually differ along. They are not the same, and it matters. |

## 4. Setup

NumPy. A synthetic "model" with a known readout, so the causal effect of an intervention
can be measured exactly — which is the property real models do not give you, and the
reason the experiments below are worth running here first.

In [1]:
# %pip install numpy

import numpy as np

rng = np.random.default_rng(0)
print("numpy", np.__version__)

numpy 2.5.1


## 5. Worked Examples

### Example 1 — build a steering vector and verify it causally

Construct activations where a "formality" concept lives along a known direction and drives
a known output, then recover and use it without being told which direction it is.

In [2]:
D_MODEL, N = 48, 3000

basis = np.linalg.qr(rng.normal(0, 1, (D_MODEL, D_MODEL)))[0]
dir_formal = basis[:, 0]         # the concept the readout depends on
dir_topic = basis[:, 1]          # an unrelated concept, for later

def generate(n, seed=0):
    r = np.random.default_rng(seed)
    formal = r.integers(0, 2, n) * 2 - 1
    topic = r.integers(0, 2, n) * 2 - 1
    h = (formal[:, None] * dir_formal * 1.0
         + topic[:, None] * dir_topic * 1.0
         + r.normal(0, 0.6, (n, D_MODEL)))
    return h, formal, topic

h, formal, topic = generate(N, seed=1)

# The "model": a readout that produces formal output when h points along dir_formal.
def model_output(h):
    return (h @ dir_formal > 0).astype(int)

# Recover the direction WITHOUT being told it -- difference in means.
v_dim = h[formal > 0].mean(0) - h[formal < 0].mean(0)
v_dim = v_dim / np.linalg.norm(v_dim)

print(f"cosine(recovered steering vector, true direction): "
      f"{float(v_dim @ dir_formal):.4f}")
print(f"baseline rate of formal output: {model_output(h).mean():.1%}\n")

print(f"{'alpha':>7} {'formal output rate':>20}")
for alpha in (-3.0, -1.5, 0.0, 1.5, 3.0):
    print(f"{alpha:7.1f} {model_output(h + alpha * v_dim).mean():20.1%}")

print("\nThe steering vector was computed from nothing but the activations and a binary")
print("label -- no gradients, no access to the readout -- and it controls the output")
print("almost completely in both directions.")
print("\nThat is a CAUSAL result in a way probe accuracy never is: the intervention")
print("changed the behaviour, so the direction is not merely readable, it is used.")

cosine(recovered steering vector, true direction): 0.9968
baseline rate of formal output: 50.6%

  alpha   formal output rate
   -3.0                 0.0%
   -1.5                10.0%
    0.0                50.6%
    1.5                89.9%
    3.0               100.0%

The steering vector was computed from nothing but the activations and a binary
label -- no gradients, no access to the readout -- and it controls the output
almost completely in both directions.

That is a CAUSAL result in a way probe accuracy never is: the intervention
changed the behaviour, so the direction is not merely readable, it is used.


### Example 2 — difference-in-means beats probe weights

A natural assumption is that a probe's weight vector is the concept's direction. It is
usually not: a probe finds *some* separating hyperplane, shaped by the noise structure of
the data, while DIM finds the axis along which the classes actually differ.

In [3]:
def fit_probe(h, y, steps=600, lr=0.5, l2=1e-3):
    w = np.zeros(h.shape[1])
    t = (y > 0).astype(float)
    for _ in range(steps):
        p = 1 / (1 + np.exp(-(h @ w)))
        w += lr * (h.T @ (t - p) / len(t) - l2 * w)
    return w

# Anisotropic noise -- large variance in some irrelevant directions, as in real models.
r = np.random.default_rng(4)
scales = np.ones(D_MODEL)
scales[2:12] = 4.0                     # a few high-variance nuisance directions
h_aniso = (formal[:, None] * dir_formal * 1.0
           + topic[:, None] * dir_topic * 1.0
           + r.normal(0, 1, (N, D_MODEL)) * scales * 0.6)

w_probe = fit_probe(h_aniso, formal); w_probe /= np.linalg.norm(w_probe)
v_dim2 = h_aniso[formal > 0].mean(0) - h_aniso[formal < 0].mean(0)
v_dim2 /= np.linalg.norm(v_dim2)

print(f"{'direction':22} {'cosine to truth':>17} {'probe test acc':>16}")
for name, vec in [("probe weights", w_probe), ("difference-in-means", v_dim2)]:
    acc = float(np.mean(((h_aniso @ vec) > 0) == (formal > 0)))
    print(f"{name:22} {float(vec @ dir_formal):17.4f} {acc:16.3f}")

print(f"\nsteering effect at matched vector norm:")
print(f"{'alpha':>7} {'via probe weights':>19} {'via difference-in-means':>25}")
for alpha in (0.0, 1.0, 2.0, 4.0):
    a = model_output(h_aniso + alpha * w_probe).mean()
    b = model_output(h_aniso + alpha * v_dim2).mean()
    print(f"{alpha:7.1f} {a:19.1%} {b:25.1%}")

print("\nNote the inversion in the first table. The PROBE classifies better -- that is")
print("what it was optimised for -- while difference-in-means is better ALIGNED with")
print("the true direction. Those are different objectives, and the probe's weights are")
print("pulled toward whichever directions best separate the classes given the noise")
print("structure, not toward the axis the concept actually varies along.")
print("\nIn this run the steering effects are close, because both vectors are decently")
print("aligned (0.94 vs 0.99). The gap widens as the nuisance directions grow, and the")
print("cheap-and-simple option is the better-aligned one -- which is why the standard")
print("recipe is DIM rather than 'train a probe and steer along its weights'.")
print("\nIt is also a concrete instance of the probing caveat: a probe finds A direction")
print("that separates the classes, not THE direction the model uses.")

direction                cosine to truth   probe test acc
probe weights                     0.9393            0.948
difference-in-means               0.9916            0.860

steering effect at matched vector norm:
  alpha   via probe weights   via difference-in-means
    0.0               49.9%                     49.9%
    1.0               72.6%                     73.9%
    2.0               90.4%                     91.5%
    4.0               99.8%                    100.0%

Note the inversion in the first table. The PROBE classifies better -- that is
what it was optimised for -- while difference-in-means is better ALIGNED with
the true direction. Those are different objectives, and the probe's weights are
pulled toward whichever directions best separate the classes given the noise
structure, not toward the axis the concept actually varies along.

In this run the steering effects are close, because both vectors are decently
aligned (0.94 vs 0.99). The gap widens as the nuisance d

### Example 3 — strength, and leaving the data manifold

The characteristic failure. Effect grows with `α` — and so does the distance from any
activation the model has ever seen.

In [4]:
# A crude but honest proxy for "is this a plausible activation": Mahalanobis-style
# distance from the reference distribution.
mu, sd = h.mean(0), h.std(0)

def off_manifold(h_steered):
    z = (h_steered - mu) / sd
    return float(np.mean(np.linalg.norm(z, axis=1)))

# A second readout: "is the output coherent" -- degrades when activations get extreme.
def coherence(h_steered):
    dist = np.linalg.norm((h_steered - mu) / sd, axis=1)
    typical = np.linalg.norm((h - mu) / sd, axis=1).mean()
    return float(np.mean(np.exp(-np.maximum(0, dist - typical * 1.3) ** 2 / 8)))

print(f"{'alpha':>7} {'formal rate':>13} {'off-manifold dist':>19} {'coherence':>11}")
for alpha in (0, 1, 2, 4, 8, 16, 32):
    hs = h + alpha * v_dim
    print(f"{alpha:7.0f} {model_output(hs).mean():13.1%} {off_manifold(hs):19.2f} "
          f"{coherence(hs):11.2f}")

print("\nThe behavioural effect saturates quickly -- by alpha=4 the output is essentially")
print("always formal and further increases buy nothing. The distance from the reference")
print("distribution, however, keeps growing without limit.")
print("\nSo everything past saturation is pure cost. In a real model that region is")
print("where you see the classic over-steering signature: the target behaviour is")
print("present but the text degenerates into repetition and broken syntax, because the")
print("activations are somewhere no training input ever put them.")
print("\nPractical rule: find the smallest alpha that achieves the effect, and treat any")
print("need for a large one as evidence your vector is poorly aligned.")

  alpha   formal rate   off-manifold dist   coherence
      0         50.6%                6.90        1.00
      1         75.6%                7.05        1.00
      2         97.7%                7.50        1.00
      4        100.0%                9.10        0.90
      8        100.0%               13.84        0.13
     16        100.0%               25.08        0.00
     32        100.0%               48.79        0.00

The behavioural effect saturates quickly -- by alpha=4 the output is essentially
always formal and further increases buy nothing. The distance from the reference
distribution, however, keeps growing without limit.

So everything past saturation is pure cost. In a real model that region is
where you see the classic over-steering signature: the target behaviour is
present but the text degenerates into repetition and broken syntax, because the
activations are somewhere no training input ever put them.

Practical rule: find the smallest alpha that achieves the effe

### Example 4 — entanglement: steering one concept moves another

The limitation that matters most in practice, and it has nothing to do with tuning. If two
concepts are correlated in the data used to build the vector, the vector contains both.

In [5]:
# Now formality and topic are CORRELATED -- as real attributes usually are.
r = np.random.default_rng(8)
n = 4000
formal_c = r.integers(0, 2, n) * 2 - 1
flip = r.random(n) < 0.15
topic_c = np.where(flip, -formal_c, formal_c)        # 85% correlated
print(f"correlation between the two concepts: "
      f"{float(np.corrcoef(formal_c, topic_c)[0, 1]):.2f}\n")

h_c = (formal_c[:, None] * dir_formal + topic_c[:, None] * dir_topic
       + r.normal(0, 0.6, (n, D_MODEL)))

def topic_output(x):
    return (x @ dir_topic > 0).astype(int)

# Naive DIM vector, built from the correlated data.
v_naive = h_c[formal_c > 0].mean(0) - h_c[formal_c < 0].mean(0)
v_naive /= np.linalg.norm(v_naive)

# A vector built with the confound BALANCED: pair up examples that share a topic.
parts = []
for t in (-1, 1):
    m = topic_c == t
    parts.append(h_c[m & (formal_c > 0)].mean(0) - h_c[m & (formal_c < 0)].mean(0))
v_balanced = np.mean(parts, axis=0)
v_balanced /= np.linalg.norm(v_balanced)

print(f"{'vector':16} {'cos to formality':>18} {'cos to topic':>14}")
for name, v in [("naive DIM", v_naive), ("balanced DIM", v_balanced)]:
    print(f"{name:16} {float(v @ dir_formal):18.3f} {float(v @ dir_topic):14.3f}")

print(f"\nsteering for FORMALITY -- what happens to the unrelated topic readout?\n")
print(f"{'alpha':>7} {'naive: formal':>15} {'naive: topic':>14} | "
      f"{'bal: formal':>13} {'bal: topic':>12}")
for alpha in (0, 1, 2, 4):
    hn, hb = h_c + alpha * v_naive, h_c + alpha * v_balanced
    print(f"{alpha:7.0f} {model_output(hn).mean():15.1%} {topic_output(hn).mean():14.1%} | "
          f"{model_output(hb).mean():13.1%} {topic_output(hb).mean():12.1%}")

print("\nThe naive vector drags the TOPIC readout along with formality, because the two")
print("were correlated in the data it was built from -- it is not a formality vector,")
print("it is a formality-and-topic vector, and nothing about the method could have")
print("revealed that.")
print("\nBalancing the confound when constructing the vector largely fixes it, and costs")
print("only care in choosing the contrast pairs. That is the practical lesson: the")
print("quality of a steering vector is decided almost entirely by the quality of the")
print("contrastive dataset, not by anything in the algorithm.")

correlation between the two concepts: 0.73

vector             cos to formality   cos to topic
naive DIM                     0.807          0.588
balanced DIM                  0.995         -0.024

steering for FORMALITY -- what happens to the unrelated topic readout?

  alpha   naive: formal   naive: topic |   bal: formal   bal: topic
      0           50.2%          50.1% |         50.2%        50.1%
      1           68.4%          61.3% |         74.5%        49.6%
      2           92.7%          79.1% |         97.7%        49.3%
      4          100.0%          99.3% |        100.0%        48.3%

The naive vector drags the TOPIC readout along with formality, because the two
were correlated in the data it was built from -- it is not a formality vector,
it is a formality-and-topic vector, and nothing about the method could have
revealed that.

Balancing the confound when constructing the vector largely fixes it, and costs
only care in choosing the contrast pairs. That is the pract

## 6. Gotchas & Pitfalls

- **Building the vector from unbalanced prompt pairs.** Example 4. Contrast pairs should
  differ in *exactly one* attribute. This is where nearly all steering-vector quality
  comes from.
- **Turning `α` up until it works.** If a large coefficient is needed, the vector is
  probably poorly aligned. Large `α` buys the target behaviour by destroying everything
  else (Example 3).
- **Using probe weights as the steering direction.** Example 2. Difference-in-means is
  both simpler and better.
- **Steering at the wrong layer.** Too early and the concept is not yet represented; too
  late and the computation that depends on it has already happened. Sweep the layers.
- **Adding the vector at every token position without thinking.** Applying it only to the
  prompt, only to generated tokens, or only at one position gives materially different
  behaviour. Decide deliberately.
- **Evaluating only the target behaviour.** Always measure something you did *not* intend
  to change; that is the only way entanglement shows up.
- **Assuming steering generalises out of distribution.** A vector built on one prompt
  format frequently fails on another. Test on held-out prompt styles.
- **Treating it as a safety guarantee.** Steering shifts a disposition; it does not
  install a constraint, and the same technique works in reverse to *remove* refusal
  behaviour.
- **Forgetting the norm.** Comparing `α` across vectors of different magnitude compares
  nothing. Normalise, or report `α·‖v‖`.

## 7. When to Use vs Alternatives

| Goal | Reach for |
|---|---|
| Test whether a direction is causally used | **Steering or ablation** — the causal counterpart to [probing](linear-probes.ipynb) |
| Cheap inference-time behaviour control | **Steering vectors** — no training, composable, reversible |
| Reliable, robust behaviour change | **Fine-tuning / DPO** — steering is not a durable control |
| Find directions without labels | [**Sparse autoencoders**](sparse-autoencoders.ipynb), then steer them to test |
| Localise *which component* causes a behaviour | Activation patching / causal tracing |
| Remove a capability | Ablation of the relevant direction — cleaner than addition, and closer to a guarantee |

**The honest position.** Steering is the cheapest causal tool in interpretability and its
main scientific value is exactly that: it converts "this direction correlates with X" into
"this direction causes X", which is the step [probes](linear-probes.ipynb) and
[SAEs](sparse-autoencoders.ipynb) cannot take on their own.

As a control mechanism it is real but limited. It works well enough to be genuinely
useful, degrades output quality as strength rises, and entangles correlated concepts in a
way no amount of tuning fixes. The refusal-direction work is the strongest existing
evidence that meaningful behaviours really are mediated by single directions — and it cuts
both ways, since the same finding shows how easily such behaviours can be removed.

Use it to *test* hypotheses about representations. Use fine-tuning when you need the
behaviour to hold.

## 8. Resources

- [Steering Language Models With Activation Engineering](https://arxiv.org/abs/2308.10248) — Turner et al.; activation addition, and the original demonstration.
- [Steering Llama 2 via Contrastive Activation Addition](https://arxiv.org/abs/2312.06681) — CAA; the difference-in-means recipe of Example 1, with layer sweeps.
- [Refusal in Language Models Is Mediated by a Single Direction](https://arxiv.org/abs/2406.11717) — the best-documented case study, including ablation as the cleaner causal test.
- [Inference-Time Intervention: Eliciting Truthful Answers from a Language Model](https://arxiv.org/abs/2306.03341) — ITI; steering toward truthfulness, and why probe directions underperform.
- [Representation Engineering: A Top-Down Approach to AI Transparency](https://arxiv.org/abs/2310.01405) — the broader framework this technique sits in.
- [Scaling Monosemanticity](https://transformer-circuits.pub/2024/scaling-monosemanticity/) — steering SAE features rather than difference-in-means directions.
- [The Linear Representation Hypothesis and the Geometry of Large Language Models](https://arxiv.org/abs/2311.03658) — the assumption underneath all of this, examined carefully.